# Lab 1d: Logistic Regression from Scratch

## Dataset: Student Performance

In this notebook you will build **Logistic Regression** completely from scratch (no
`scikit-learn`) using only `numpy`, `pandas` and `matplotlib`.

**Goal:** predict `Pass_Fail` (1 = pass, 0 = fail) from `Study_Hours`, `Attendance` and
`Practice_Tests`. This is a **classification** problem, so it needs a different model
function and cost function than linear regression - but, remarkably, the very same
gradient descent loop.

## Notebooks in this lab
| | from scratch | scikit-learn |
|---|---|---|
| **Linear Regression** | `linear_regression_scratch.ipynb` | `linear_regression_sklearn.ipynb` |
| **Logistic Regression** | **`logistic_regression_scratch.ipynb`** (this one) | `logistic_regression_sklearn.ipynb` |

## Outline
- [1 - Packages](#1)
- [2 - Load the Dataset](#2)
- [3 - Explore & Prepare the Data](#3)
  - [3.1 Data types](#3.1)
  - [3.2 Train / Test split with pandas](#3.2)
  - [3.3 Feature normalization](#3.3)
- [4 - Sigmoid & model function](#4)
- [5 - Cost function](#5)
- [6 - Gradient descent](#6)
- [7 - Training](#7)
- [8 - Results](#8)

<a name="1"></a>
## 1 - Packages

- [numpy](https://www.numpy.org) for vectorized math
- [pandas](https://pandas.pydata.org) to load and manipulate the dataset
- [matplotlib](https://matplotlib.org) to plot the loss and the results

In [ ]:
# On Kaggle / Colab: get the lab files. Skip this cell if you already run the notebook
# from inside the repository folder.
!git clone https://github.com/MLs-labs/Lab_1_ML

In [ ]:
import os
import sys

# Works both on Kaggle (after the git clone above) and locally from the repo folder.
REPO_DIR = "/kaggle/working/Lab_1_ML"
if not os.path.isdir(REPO_DIR):
    REPO_DIR = "."
sys.path.append(REPO_DIR)

import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# public_tests.py contains the checks used throughout this notebook
importlib.invalidate_caches()
from public_tests import *

np.random.seed(1)

<a name="2"></a>
## 2 - Load the Dataset

The dataset `studentPerformance.csv` contains, for each student:
- `Study_Hours`: number of hours studied per day
- `Attendance`: attendance percentage
- `Practice_Tests`: number of practice tests taken
- `Final_Score`: final exam score (0-100) &rarr; **target for Linear Regression**
- `Pass_Fail`: 1 if the student passed, 0 otherwise &rarr; **target for Logistic Regression**

We load it with `pandas.read_csv`, which is the standard way to read a CSV file into a
`DataFrame`.

In [ ]:
df = pd.read_csv(os.path.join(REPO_DIR, "studentPerformance.csv"))

# Always look at your data before doing anything else.
print("Shape of the dataset (rows, columns):", df.shape)
df.head()

<a name="3"></a>
## 3 - Explore & Prepare the Data

<a name="3.1"></a>
### 3.1 Data types

Before building a model, you need to know what you are working with. `df.dtypes` tells you
the type pandas inferred for every column (`int64`, `float64`, `object`, ...). This matters
because a model only understands numbers, so any non-numeric column would need to be
encoded first.

In [ ]:
print(df.dtypes)

# `df.info()` gives a more complete summary: dtypes, non-null counts and memory usage.
df.info()

# Check for missing values.
print("\nMissing values per column:\n", df.isnull().sum())

# Quick statistical summary (mean, std, min, max, quartiles) for every numeric column.
df.describe()

All 5 columns are numeric (`int64` or `float64`) and there are no missing values, so the
dataset is already usable as-is - no encoding or imputation is required.

<a name="3.2"></a>
### 3.2 Train / Test split with pandas

We need a **training set** to fit the model and a separate **test set** to check that it
generalizes to data it has never seen. A simple and common way to do this with pandas is:

1. Shuffle the rows of the DataFrame with `df.sample(frac=1, ...)`
2. Cut the shuffled DataFrame at a chosen index (e.g. 80% train / 20% test) using `.iloc`

`frac=1` means "sample 100% of the rows", which shuffles the whole DataFrame.
`random_state` makes the shuffle reproducible.

In [ ]:
train_ratio = 0.8

### START CODE HERE ### (~ 5 lines of code)
# shuffle df with df.sample(frac=1, random_state=1) and reset_index(drop=True)
df_shuffled = None
# index that marks 80% of the way through df_shuffled
split_index = None
# slice df_shuffled with .iloc to get the train and test DataFrames
df_train = None
df_test = None
### END CODE HERE ###

print(f"Training examples: {len(df_train)}")
print(f"Test examples:     {len(df_test)}")

# Now split each set into input features `X` and targets `y`, and convert them to numpy
# arrays since our from-scratch functions work with numpy, not DataFrames.
feature_cols = ["Study_Hours", "Attendance", "Practice_Tests"]

X_train = df_train[feature_cols].to_numpy()
X_test = df_test[feature_cols].to_numpy()

y_train_clf = df_train["Pass_Fail"].to_numpy()
y_test_clf = df_test["Pass_Fail"].to_numpy()

print("X_train shape:", X_train.shape)
print("y_train_clf shape:", y_train_clf.shape)

Run the cell below to check that your train/test split is correct.

In [ ]:
split_test(df, df_train, df_test, X_train, X_test)

<a name="3.3"></a>
### 3.3 Feature normalization

The three features live on very different scales (hours vs. percentage vs. test count).
Gradient descent converges much faster and more reliably when features are on a similar
scale, so we apply **z-score normalization**:
$$ x_{norm} = \frac{x - \mu}{\sigma} $$
The mean `mu` and standard deviation `sigma` are computed **only on the training set** and
then reused to scale the test set, so that no information from the test set leaks into
training.

In [ ]:
def zscore_normalize(X, mu=None, sigma=None):
    """Normalizes the columns of X to zero mean and unit variance."""
    if mu is None or sigma is None:
        mu = np.mean(X, axis=0)
        sigma = np.std(X, axis=0)
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

X_train_norm, mu, sigma = zscore_normalize(X_train)
X_test_norm, _, _ = zscore_normalize(X_test, mu, sigma)

print("Feature means (train):", mu)
print("Feature stds  (train):", sigma)

<a name="4"></a>
## 4 - Sigmoid & model function

Logistic regression squashes the linear combination $w \cdot x + b$ through the **sigmoid**
function so that the output can be interpreted as a probability between 0 and 1:
$$ g(z) = \frac{1}{1 + e^{-z}}, \qquad f_{w,b}(x) = g(w \cdot x + b) $$

In [ ]:
def sigmoid(z):
    """Computes the sigmoid of z, element-wise."""
    return 1 / (1 + np.exp(-z))

def predict_logistic(X, w, b):
    """
    Computes the logistic regression prediction (probability of class 1) for every row of X.

    Args:
        X (ndarray (m,n)): m examples, n features
        w (ndarray (n,)):  weights
        b (scalar):        bias

    Returns:
        ndarray (m,): predicted probabilities f_wb(x) for every example
    """
    z = X @ w + b
    return sigmoid(z)

A quick look at the sigmoid curve:

In [ ]:
z = np.linspace(-8, 8, 200)
plt.figure(figsize=(6, 4))
plt.plot(z, sigmoid(z))
plt.axhline(0.5, color="k", linestyle="--", linewidth=1)
plt.title("Sigmoid function")
plt.xlabel("z")
plt.ylabel("g(z)")
plt.show()

<a name="5"></a>
## 5 - Cost function

For classification we use the **log loss** (binary cross-entropy), which penalizes confident
wrong predictions much more than the squared error would:
$$ J(w,b) = -\frac{1}{m} \sum_{i=0}^{m-1} \left[ y^{(i)} \log\left(f_{w,b}(x^{(i)})\right) +
      (1-y^{(i)}) \log\left(1-f_{w,b}(x^{(i)})\right) \right] $$

In [ ]:
def compute_cost_logistic(X, y, w, b):
    """
    Computes the log loss (binary cross-entropy) cost for logistic regression.

    Args:
        X (ndarray (m,n)): m examples, n features
        y (ndarray (m,)):  actual labels (0 or 1)
        w (ndarray (n,)):  weights
        b (scalar):        bias

    Returns:
        total_cost (float): the cost J(w,b)
    """
    m = X.shape[0]
    eps = 1e-12  # avoids log(0)

    ### START CODE HERE ### (~ 2 lines of code)
    f_wb = None
    total_cost = None
    ### END CODE HERE ###

    return total_cost

# Sanity check with w = 0, b = 0 -> f_wb = 0.5 for every example
w_init = np.zeros(X_train_norm.shape[1])
b_init = 0.0
print("Cost at w=0, b=0:", compute_cost_logistic(X_train_norm, y_train_clf, w_init, b_init))

Run the cell below to check your `compute_cost_logistic` implementation.

In [ ]:
compute_cost_logistic_test(compute_cost_logistic)

<a name="6"></a>
## 6 - Gradient descent

Remarkably, the gradient of the logistic regression cost has the **exact same form** as for
linear regression - only `f_wb` is now computed with the sigmoid:
$$ \frac{\partial J(w,b)}{\partial w_j} = \frac{1}{m} \sum_{i=0}^{m-1}
      \left( f_{w,b}(x^{(i)}) - y^{(i)} \right) x_j^{(i)} $$
$$ \frac{\partial J(w,b)}{\partial b} = \frac{1}{m} \sum_{i=0}^{m-1}
      \left( f_{w,b}(x^{(i)}) - y^{(i)} \right) $$

The `gradient_descent` loop below is identical to the one in
`linear_regression_scratch.ipynb`: only the cost and gradient functions you pass to it
change.

In [ ]:
def compute_gradient_logistic(X, y, w, b):
    """
    Computes the gradient of the logistic regression cost with respect to w and b.

    Args:
        X (ndarray (m,n)): m examples, n features
        y (ndarray (m,)):  actual labels (0 or 1)
        w (ndarray (n,)):  weights
        b (scalar):        bias

    Returns:
        dj_dw (ndarray (n,)): gradient of the cost w.r.t. w
        dj_db (scalar):       gradient of the cost w.r.t. b
    """
    m = X.shape[0]

    ### START CODE HERE ### (~ 3 lines of code)
    error = None      # f_wb - y, shape (m,)
    dj_dw = None       # shape (n,)
    dj_db = None
    ### END CODE HERE ###

    return dj_dw, dj_db

def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters):
    """
    Performs batch gradient descent to fit w, b.

    Args:
        X, y :                 training data and targets
        w_in, b_in :           initial values of the parameters
        cost_function :        function to compute the cost
        gradient_function :    function to compute the gradient
        alpha (float):         learning rate
        num_iters (int):       number of iterations

    Returns:
        w, b :        parameters found after running gradient descent
        J_history :   cost at every iteration (for plotting)
    """
    w = w_in.copy()
    b = b_in
    J_history = []

    for i in range(num_iters):
        dj_dw, dj_db = gradient_function(X, y, w, b)

        ### START CODE HERE ### (~ 2 lines of code)
        # simultaneously update w and b using the gradient and the learning rate
        w = None
        b = None
        ### END CODE HERE ###

        J_history.append(cost_function(X, y, w, b))

        if i % max(1, num_iters // 10) == 0:
            print(f"Iteration {i:5}: Cost {J_history[-1]:8.4f}")

    return w, b, J_history

Run the cell below to check your `compute_gradient_logistic` implementation and the
parameter-update step inside `gradient_descent`.

In [ ]:
compute_gradient_logistic_test(compute_gradient_logistic, gradient_descent, compute_cost_logistic)

<a name="7"></a>
## 7 - Training

We run gradient descent on the **normalized training features**, with the pass/fail labels
as targets.

In [ ]:
alpha = 0.5
num_iters = 1000

w_init = np.zeros(X_train_norm.shape[1])
b_init = 0.0

### START CODE HERE ### (~ 1 line of code)
# call gradient_descent with X_train_norm, y_train_clf, w_init, b_init,
# compute_cost_logistic, compute_gradient_logistic, alpha and num_iters
w_final_clf, b_final_clf, J_hist_logistic = None, None, None
### END CODE HERE ###

print("\nw,b found by gradient descent:", w_final_clf, b_final_clf)

Run the cell below to check that training on the real dataset converges to the expected
parameters and reaches a good test accuracy.

In [ ]:
train_logistic_test(w_final_clf, b_final_clf, J_hist_logistic, predict_logistic, X_test_norm, y_test_clf)

<a name="8"></a>
## 8 - Results

#### Plot the loss

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(J_hist_logistic)
plt.title("Logistic Regression - Cost vs. Iteration")
plt.xlabel("Iteration")
plt.ylabel("Cost J(w,b)")
plt.show()

#### Evaluate on the test set

We turn the predicted probability into a class prediction by thresholding at 0.5, then
compute the accuracy: the fraction of correctly classified students.

In [ ]:
probs_test = predict_logistic(X_test_norm, w_final_clf, b_final_clf)
y_pred_clf = (probs_test >= 0.5).astype(int)

accuracy = np.mean(y_pred_clf == y_test_clf)
print(f"Test accuracy: {accuracy * 100:.2f}%")

test_cost_clf = compute_cost_logistic(X_test_norm, y_test_clf, w_final_clf, b_final_clf)
print(f"Cost on the test set: {test_cost_clf:.4f}")

#### Visualize predicted probabilities

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(range(len(probs_test)), probs_test, c=y_test_clf, cmap="coolwarm", alpha=0.7)
plt.axhline(0.5, color="k", linestyle="--", label="decision threshold")
plt.title("Logistic Regression - Predicted Probability of Passing (test set)")
plt.xlabel("Test example index")
plt.ylabel("Predicted P(Pass)")
plt.legend()
plt.show()

**Congratulations!** You built Logistic Regression completely from scratch: the sigmoid, the
log loss, its gradient and the training loop, plus an evaluation on a held-out test set.

Compare with `logistic_regression_sklearn.ipynb` to see how much of this scikit-learn does
for you in a single `.fit()` call.